In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 1 — Install dependencies                               ║
# ╚══════════════════════════════════════════════════════════════╝
!pip install transformers==5.2.0 peft==0.18.1 sentencepiece \
             sacrebleu rouge-score nltk jiwer joblib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 72.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.0 MB/s eta 0:00:00:00:01


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 2 — Configuration                                      ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Model paths ───────────────────────────────────────────────────────────────
CLASSIFIER_DIR  = "/kaggle/input/datasets/austraa/bangla-bert-dialect"
BASE_MODEL_PATH = "/kaggle/input/datasets/farhanaadri/nllb-dialect-model"

# Adapter dataset paths — one per dialect
# Each path should point to the folder containing adapter_config.json
ADAPTER_PATHS = {
    "chittagong" : "/kaggle/input/datasets/mahbubasayedsinthia/chittagong-adapter-hp",
    "sylhet"     : "/kaggle/input/datasets/mahbubasayedsinthia/sylhet-adapter-hp",
    "noakhali"   : "/kaggle/input/datasets/mahbubasayedsinthia/noakhali-adapter-hp",
    "barisal"    : "/kaggle/input/datasets/mahbubasayedsinthia/barisal-adapter-hp",
    "mymensingh" : "/kaggle/input/datasets/abonykamal/mymensingh-adapter-hp",
}

# ── Dialect metadata ──────────────────────────────────────────────────────────
# Maps BanglaBERT output label → (adapter_name, NLLB src_lang_tag)
# Standard Bangla has no adapter → adapter_name is None
DIALECT_CONFIG = {
    "Chittagong"     : {"adapter": "chittagong",  "src_lang": "bng_chittagong"},
    "Sylhet"         : {"adapter": "sylhet",       "src_lang": "bng_sylhet"},
    "Noakhali"       : {"adapter": "noakhali",     "src_lang": "bng_noakhali"},
    "Barisal"        : {"adapter": "barisal",      "src_lang": "bng_barisal"},
    "Mymensingh"     : {"adapter": "mymensingh",   "src_lang": "bng_mymensingh"},
    "Standard Bangla": {"adapter": None,           "src_lang": "ben_Beng"},
}

# ── Generation hyperparameters ────────────────────────────────────────────────
# Using the winning configs per dialect from your HP search
GEN_CONFIG = {
    "chittagong" : {"num_beams": 3,  "length_penalty": 0.8},
    "sylhet"     : {"num_beams": 8,  "length_penalty": 1.0},
    "noakhali"   : {"num_beams": 3,  "length_penalty": 0.8},
    "barisal"    : {"num_beams": 3,  "length_penalty": 0.8},
    "mymensingh" : {"num_beams": 3,  "length_penalty": 0.8},
    "standard"   : {"num_beams": 4,  "length_penalty": 1.0},
}

MAX_SOURCE_LEN = 128
MAX_TARGET_LEN = 128
TGT_LANG       = "eng_Latn"

# ── Confidence threshold ──────────────────────────────────────────────────────
# If classifier confidence is below this, log a warning but still proceed
CONFIDENCE_THRESHOLD = 0.60

print("Configuration loaded.")

Configuration loaded.


In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 3 — Imports and normalization utilities                ║
# ╚══════════════════════════════════════════════════════════════╝
import os, re, unicodedata, warnings
import numpy as np
import torch
import joblib
from contextlib import contextmanager

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    NllbTokenizer,
    AutoModelForSeq2SeqLM,
)
from peft import PeftModel

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}  |  GPU: {torch.cuda.get_device_name(0) if device == 'cuda' else 'N/A'}")


# ── Text normalization ────────────────────────────────────────────────────────
# Two functions: one for the classifier (aggressive, matches training),
# one for NLLB (dialect-safe, whitespace/punctuation only).

def normalize_for_classifier(text: str) -> str:
    """
    Matches the normalization used during BanglaBERT training.
    Aggressive: NFKC, zero-width removal, lowercase.
    """
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = text.replace("৷", "।")
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


_PUNCT_MAP = str.maketrans({
    "\u201c": '"', "\u201d": '"',
    "\u2018": "'", "\u2019": "'",
    "\u2013": "-", "\u2014": "-",
    "\u2026": "...",
    "\u00a0": " ",
    "\u200b": "", "\u200c": "", "\u200d": "", "\ufeff": "",
})

def normalize_for_translation(text: str) -> str:
    """
    Dialect-safe normalization for NLLB input.
    Whitespace and punctuation only — preserves dialectal features.
    """
    if not isinstance(text, str):
        return ""
    text = text.translate(_PUNCT_MAP)
    text = re.sub(r"[\u200b-\u200f\ufeff]", "", text)
    text = re.sub(r" +", " ", text).strip()
    return text


print("Normalization functions ready.")

Device: cuda  |  GPU: Tesla T4
Normalization functions ready.


In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 4 — Load BanglaBERT dialect classifier                 ║
# ╚══════════════════════════════════════════════════════════════╝

print(f"Loading classifier from: {CLASSIFIER_DIR}")

cls_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_DIR)
cls_model     = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_DIR)
label_encoder = joblib.load(os.path.join(CLASSIFIER_DIR, "label_encoder.joblib"))

cls_model.to(device)
cls_model.eval()

print(f"  Classes : {list(label_encoder.classes_)}")
print(f"  Params  : {sum(p.numel() for p in cls_model.parameters()):,}")
print("Classifier ready.")

Loading classifier from: /kaggle/input/datasets/austraa/bangla-bert-dialect


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Classes : ['Barisal', 'Chittagong', 'Mymensingh', 'Noakhali', 'Standard Bangla', 'Sylhet']
  Params  : 110,621,958
Classifier ready.


In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 5 — Load NLLB base model + all 5 LoRA adapters        ║
# ╚══════════════════════════════════════════════════════════════╝
# Strategy: load all adapters into the model at startup using PEFT's
# multi-adapter API. Hot-swapping with set_adapter() is then ~instant.
# Standard Bangla uses disable_adapter() to run the frozen base model.

print(f"Loading NLLB base model from: {BASE_MODEL_PATH}")
nllb_tokenizer = NllbTokenizer.from_pretrained(BASE_MODEL_PATH, use_fast=False)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ignore_mismatched_sizes=True,
)
print(f"  Base model params: {sum(p.numel() for p in base_model.parameters()):,}")

# Load the first adapter to initialise the PeftModel wrapper
first_dialect = list(ADAPTER_PATHS.keys())[0]
print(f"\nLoading adapter: {first_dialect}")
nllb_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATHS[first_dialect],
    adapter_name=first_dialect,
)

# Load the remaining adapters into the same PeftModel
for dialect, path in list(ADAPTER_PATHS.items())[1:]:
    print(f"Loading adapter: {dialect}")
    nllb_model.load_adapter(path, adapter_name=dialect)

nllb_model.to(device)
nllb_model.eval()

# Sanity check: list loaded adapters
loaded = list(nllb_model.peft_config.keys())
print(f"\nAll adapters loaded: {loaded}")

# Verify dialect tags exist in tokenizer
for dialect, cfg in DIALECT_CONFIG.items():
    tag_id = nllb_tokenizer.convert_tokens_to_ids(cfg["src_lang"])
    status = "✅" if tag_id != nllb_tokenizer.unk_token_id else "❌ MISSING"
    print(f"  {cfg['src_lang']:20s} → token_id {tag_id}  {status}")

print("\nNLLB model + all adapters ready.")

Loading NLLB base model from: /kaggle/input/datasets/farhanaadri/nllb-dialect-model


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Base model params: 615,076,864

Loading adapter: chittagong
Loading adapter: sylhet
Loading adapter: noakhali
Loading adapter: barisal
Loading adapter: mymensingh

All adapters loaded: ['chittagong', 'sylhet', 'noakhali', 'barisal', 'mymensingh']
  bng_chittagong       → token_id 256207  ✅
  bng_sylhet           → token_id 256208  ✅
  bng_noakhali         → token_id 256206  ✅
  bng_barisal          → token_id 256204  ✅
  bng_mymensingh       → token_id 256205  ✅
  ben_Beng             → token_id 256026  ✅

NLLB model + all adapters ready.


In [6]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 6 — Inference pipeline class                           ║
# ╚══════════════════════════════════════════════════════════════╝

class BengaliDialectTranslator:
    """
    End-to-end pipeline: Bengali dialect text → English translation.

    Steps:
      1. Classify dialect with BanglaBERT
      2. Select the correct LoRA adapter (or disable for Standard Bangla)
      3. Translate with NLLB + selected adapter
    """

    def __init__(
        self,
        cls_model,
        cls_tokenizer,
        label_encoder,
        nllb_model,
        nllb_tokenizer,
        dialect_config,
        gen_config,
        device,
        max_source_len=128,
        max_target_len=128,
        confidence_threshold=0.60,
    ):
        self.cls_model    = cls_model
        self.cls_tok      = cls_tokenizer
        self.le           = label_encoder
        self.nllb         = nllb_model
        self.nllb_tok     = nllb_tokenizer
        self.dialect_cfg  = dialect_config
        self.gen_cfg      = gen_config
        self.device       = device
        self.max_src      = max_source_len
        self.max_tgt      = max_target_len
        self.conf_thresh  = confidence_threshold
        self.tgt_lang_id  = nllb_tokenizer.convert_tokens_to_ids(TGT_LANG)

    # ── Step 1: classify ──────────────────────────────────────────────────────
    def classify(self, text: str) -> dict:
        """Returns dialect label, confidence, and full probability distribution."""
        clean = normalize_for_classifier(text)
        inputs = self.cls_tok(
            clean,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128,
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = self.cls_model(**inputs).logits
            probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()

        pred_id    = int(np.argmax(probs))
        pred_label = self.le.inverse_transform([pred_id])[0]
        confidence = float(probs[pred_id])

        all_probs = {
            self.le.inverse_transform([i])[0]: float(p)
            for i, p in enumerate(probs)
        }

        if confidence < self.conf_thresh:
            print(
                f"  ⚠️  Low confidence ({confidence:.2%}) for '{pred_label}'. "
                f"Result may be unreliable."
            )

        return {
            "dialect"    : pred_label,
            "confidence" : confidence,
            "all_probs"  : all_probs,
        }

    # ── Step 2: select adapter ────────────────────────────────────────────────
    @contextmanager
    def _adapter_context(self, dialect_label: str):
        """
        Context manager that activates the right adapter for the given dialect,
        then restores state on exit. Thread-unsafe but fine for single-process
        Kaggle inference.
        """
        cfg          = self.dialect_cfg[dialect_label]
        adapter_name = cfg["adapter"]

        if adapter_name is None:
            # Standard Bangla: run the frozen base model with no adapter
            with self.nllb.disable_adapter():
                yield cfg
        else:
            self.nllb.set_adapter(adapter_name)
            yield cfg

    # ── Step 3: translate ─────────────────────────────────────────────────────
    def translate(self, text: str, dialect_label: str) -> str:
        """
        Translates a single string using the adapter for the specified dialect.
        You can call this directly if you already know the dialect.
        """
        cfg        = self.dialect_cfg[dialect_label]
        gen_params = self.gen_cfg.get(
            cfg["adapter"] or "standard",
            {"num_beams": 4, "length_penalty": 1.0}
        )
        clean = normalize_for_translation(text)

        with self._adapter_context(dialect_label):
            self.nllb_tok.src_lang = cfg["src_lang"]
            inputs = self.nllb_tok(
                clean,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=self.max_src,
            ).to(self.device)

            with torch.no_grad():
                generated = self.nllb.generate(
                    **inputs,
                    forced_bos_token_id = self.tgt_lang_id,
                    max_length          = self.max_tgt,
                    num_beams           = gen_params["num_beams"],
                    length_penalty      = gen_params["length_penalty"],
                    early_stopping      = True,
                )

        translation = self.nllb_tok.decode(generated[0], skip_special_tokens=True)
        return translation

    # ── Full pipeline: classify + translate ───────────────────────────────────
    def __call__(self, text: str, return_details: bool = False):
        """
        Full pipeline. Returns translation string by default.
        Set return_details=True to get the full result dict.
        """
        cls_result  = self.classify(text)
        dialect     = cls_result["dialect"]
        translation = self.translate(text, dialect)

        result = {
            "input"       : text,
            "dialect"     : dialect,
            "confidence"  : round(cls_result["confidence"], 4),
            "translation" : translation,
            "all_probs"   : {k: round(v, 4) for k, v in cls_result["all_probs"].items()},
        }

        if return_details:
            return result
        return translation

    # ── Batch inference ───────────────────────────────────────────────────────
    def translate_batch(self, texts: list[str], batch_size: int = 16) -> list[dict]:
        """
        Classifies and translates a list of texts.
        Groups by dialect to minimise adapter switches (1 switch per dialect group).
        """
        # Step 1: classify all texts
        print(f"Classifying {len(texts)} texts...")
        classified = []
        for text in texts:
            cls = self.classify(text)
            classified.append((text, cls["dialect"], cls["confidence"]))

        # Step 2: group indices by dialect to batch same-dialect inputs together
        from collections import defaultdict
        dialect_groups = defaultdict(list)
        for i, (text, dialect, conf) in enumerate(classified):
            dialect_groups[dialect].append(i)

        print(f"Dialect distribution: { {d: len(idxs) for d, idxs in dialect_groups.items()} }")

        # Step 3: translate each group in batches
        translations = [""] * len(texts)

        for dialect, indices in dialect_groups.items():
            cfg        = self.dialect_cfg[dialect]
            gen_params = self.gen_cfg.get(
                cfg["adapter"] or "standard",
                {"num_beams": 4, "length_penalty": 1.0}
            )
            print(f"  Translating {len(indices)} {dialect} samples "
                  f"(beams={gen_params['num_beams']})...")

            with self._adapter_context(dialect):
                self.nllb_tok.src_lang = cfg["src_lang"]

                for start in range(0, len(indices), batch_size):
                    batch_indices = indices[start : start + batch_size]
                    batch_texts   = [
                        normalize_for_translation(classified[i][0])
                        for i in batch_indices
                    ]

                    inputs = self.nllb_tok(
                        batch_texts,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=self.max_src,
                    ).to(self.device)

                    with torch.no_grad():
                        generated = self.nllb.generate(
                            **inputs,
                            forced_bos_token_id = self.tgt_lang_id,
                            max_length          = self.max_tgt,
                            num_beams           = gen_params["num_beams"],
                            length_penalty      = gen_params["length_penalty"],
                            early_stopping      = True,
                        )

                    decoded = self.nllb_tok.batch_decode(
                        generated, skip_special_tokens=True
                    )
                    for j, idx in enumerate(batch_indices):
                        translations[idx] = decoded[j]

        # Assemble results
        return [
            {
                "input"      : classified[i][0],
                "dialect"    : classified[i][1],
                "confidence" : round(classified[i][2], 4),
                "translation": translations[i],
            }
            for i in range(len(texts))
        ]


print("Pipeline class defined.")

Pipeline class defined.


In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 7 — Instantiate the pipeline                           ║
# ╚══════════════════════════════════════════════════════════════╝

pipeline = BengaliDialectTranslator(
    cls_model             = cls_model,
    cls_tokenizer         = cls_tokenizer,
    label_encoder         = label_encoder,
    nllb_model            = nllb_model,
    nllb_tokenizer        = nllb_tokenizer,
    dialect_config        = DIALECT_CONFIG,
    gen_config            = GEN_CONFIG,
    device                = device,
    max_source_len        = MAX_SOURCE_LEN,
    max_target_len        = MAX_TARGET_LEN,
    confidence_threshold  = CONFIDENCE_THRESHOLD,
)

print("Pipeline ready ✅")

Pipeline ready ✅


In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 8 — Single-sample inference with details               ║
# ╚══════════════════════════════════════════════════════════════╝

test_sentences = [
    "আঁই হন্ডে যাইচ",           # Chittagong
    "তুমি কিতা করো আসিলা",       # Sylhet
    "তুঁই কই যাইবু",             # Noakhali
    "তুমি কই যাইতাছ",            # Barisal
    "তুমি কই যাও",               # Mymensingh
    "আপনি কেমন আছেন",            # Standard Bangla
]

print("=" * 60)
for sentence in test_sentences:
    result = pipeline(sentence, return_details=True)
    print(f"Input      : {result['input']}")
    print(f"Dialect    : {result['dialect']}  (confidence: {result['confidence']:.2%})")
    print(f"Translation: {result['translation']}")
    print("-" * 60)

Input      : আঁই হন্ডে যাইচ
Dialect    : Chittagong  (confidence: 99.84%)
Translation: Where should I go?
------------------------------------------------------------
Input      : তুমি কিতা করো আসিলা
Dialect    : Mymensingh  (confidence: 99.53%)
Translation: What were you doing?
------------------------------------------------------------
Input      : তুঁই কই যাইবু
Dialect    : Chittagong  (confidence: 99.75%)
Translation: Where will you go?
------------------------------------------------------------
Input      : তুমি কই যাইতাছ
Dialect    : Mymensingh  (confidence: 98.37%)
Translation: Where are you going?
------------------------------------------------------------
Input      : তুমি কই যাও
Dialect    : Sylhet  (confidence: 98.89%)
Translation: Where are you going?
------------------------------------------------------------
Input      : আপনি কেমন আছেন
Dialect    : Standard Bangla  (confidence: 99.72%)
Translation: How are you?
---------------------------------------------------------

In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 9 (updated) — Batch inference on combined test CSV     ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd
from pathlib import Path

# Combined file produced by Cell B above
TEST_CSV = "/kaggle/input/datasets/mahbubasayedsinthia/dataset-test-dialect/combined_test.csv"

df_test = pd.read_csv(TEST_CSV)
texts   = df_test["dialect_speech"].tolist()
refs    = df_test["english_translation"].tolist()

# The 'dialect' column lets you check how often the pipeline
# agrees with the ground-truth dialect label
ground_truth_dialects = df_test["dialect"].tolist()

results = pipeline.translate_batch(texts, batch_size=16)
df_out  = pd.DataFrame(results)

df_out["reference"]              = refs
df_out["ground_truth_dialect"]   = ground_truth_dialects

# Flag rows where the classifier disagreed with the known dialect
df_out["dialect_match"] = df_out["dialect"] == df_out["ground_truth_dialect"]

output_path = Path("/kaggle/working/pipeline_predictions.csv")
df_out.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Saved {len(df_out)} predictions to {output_path}")
print(f"\nClassifier agreement with ground-truth dialect labels:")
print(df_out.groupby("ground_truth_dialect")["dialect_match"].mean().round(3).to_string())
print(f"\nOverall dialect accuracy: {df_out['dialect_match'].mean():.2%}")
print(df_out[["input", "ground_truth_dialect", "dialect",         # ← was "dialect_speech"
              "dialect_match", "translation"]].head(8).to_string())

Classifying 660 texts...
  ⚠️  Low confidence (54.38%) for 'Noakhali'. Result may be unreliable.
  ⚠️  Low confidence (44.43%) for 'Noakhali'. Result may be unreliable.
  ⚠️  Low confidence (59.15%) for 'Mymensingh'. Result may be unreliable.
  ⚠️  Low confidence (58.09%) for 'Mymensingh'. Result may be unreliable.
  ⚠️  Low confidence (59.96%) for 'Noakhali'. Result may be unreliable.
  ⚠️  Low confidence (51.86%) for 'Barisal'. Result may be unreliable.
  ⚠️  Low confidence (59.51%) for 'Barisal'. Result may be unreliable.
  ⚠️  Low confidence (57.53%) for 'Barisal'. Result may be unreliable.
  ⚠️  Low confidence (54.25%) for 'Mymensingh'. Result may be unreliable.
Dialect distribution: {'Noakhali': 139, 'Chittagong': 128, 'Sylhet': 122, 'Mymensingh': 135, 'Barisal': 131, 'Standard Bangla': 5}
  Translating 139 Noakhali samples (beams=3)...
  Translating 128 Chittagong samples (beams=3)...
  Translating 122 Sylhet samples (beams=8)...
  Translating 135 Mymensingh samples (beams=3)...

In [12]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 10 — Optional: evaluate pipeline predictions           ║
# ╚══════════════════════════════════════════════════════════════╝
# Only run this if you have reference translations (refs is not None)
import sacrebleu
import numpy as np

if refs:
    preds_clean = [r.strip() for r in df_out["translation"].tolist()]
    refs_clean  = [r.strip() for r in refs]

    bleu  = sacrebleu.corpus_bleu(preds_clean, [refs_clean]).score
    chrf  = sacrebleu.corpus_chrf(preds_clean, [refs_clean], word_order=2).score

    print(f"Pipeline SacreBLEU : {bleu:.2f}")
    print(f"Pipeline chrF++    : {chrf:.2f}")

    # Per-dialect breakdown
    print("\nPer-dialect BLEU:")
    for dialect in df_out["dialect"].unique():
        mask  = df_out["dialect"] == dialect
        p_sub = [preds_clean[i] for i, m in enumerate(mask) if m]
        r_sub = [refs_clean[i]  for i, m in enumerate(mask) if m]
        if p_sub:
            b = sacrebleu.corpus_bleu(p_sub, [r_sub]).score
            print(f"  {dialect:20s}: {b:.2f}  (n={len(p_sub)})")

Pipeline SacreBLEU : 43.37
Pipeline chrF++    : 64.81

Per-dialect BLEU:
  Noakhali            : 39.09  (n=139)
  Chittagong          : 40.30  (n=128)
  Sylhet              : 40.76  (n=122)
  Mymensingh          : 48.71  (n=135)
  Barisal             : 47.89  (n=131)
  Standard Bangla     : 41.74  (n=5)


In [13]:
# Run this on df_out from Cell 10
print("Classifier accuracy per dialect:")
print(df_out.groupby("ground_truth_dialect")["dialect_match"].mean().round(3).to_string())

# See what dialects are being confused with what
confusion = df_out.groupby(["ground_truth_dialect", "dialect"]).size().unstack(fill_value=0)
print("\nConfusion matrix (rows=ground truth, cols=predicted):")
print(confusion.to_string())

Classifier accuracy per dialect:
ground_truth_dialect
Barisal       0.992
Chittagong    0.977
Mymensingh    0.985
Noakhali      0.985
Sylhet        0.890

Confusion matrix (rows=ground truth, cols=predicted):
dialect               Barisal  Chittagong  Mymensingh  Noakhali  Standard Bangla  Sylhet
ground_truth_dialect                                                                    
Barisal                   130           0           1         0                0       0
Chittagong                  0         127           0         3                0       0
Mymensingh                  0           0         128         0                1       1
Noakhali                    0           1           1       131                0       0
Sylhet                      1           0           5         5                4     121


In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell M1 — Install additional dependencies                   ║
# ╚══════════════════════════════════════════════════════════════╝
!pip install bert-score unbabel-comet --quiet
# sacrebleu already installed — TER is built into it

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 89.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.3 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [16]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell M2 — Load predictions (output of Cell 9)               ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd
import numpy as np

df = pd.read_csv("/kaggle/working/pipeline_predictions.csv")

# Normalise column names in case of slight differences
df.columns = df.columns.str.strip()

preds = df["translation"].str.strip().tolist()
refs  = df["reference"].str.strip().tolist()
dialects = df["ground_truth_dialect"].tolist()

print(f"Loaded {len(df)} predictions")
print(f"Dialects present: {sorted(df['ground_truth_dialect'].unique())}")

Loaded 660 predictions
Dialects present: ['Barisal', 'Chittagong', 'Mymensingh', 'Noakhali', 'Sylhet']


In [17]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell M3 — TER (Translation Edit Rate)                       ║
# ║  Measures word-level edit distance normalised by ref length.  ║
# ║  Complements WER — TER allows block shifts (reorderings)     ║
# ║  whereas WER does not. Lower is better.                      ║
# ╚══════════════════════════════════════════════════════════════╝
import sacrebleu

ter_result = sacrebleu.corpus_ter(preds, [refs])
ter_score  = ter_result.score

print(f"TER (corpus): {ter_score:.2f}")

# Per-sentence TER for later use
sent_ter = [
    sacrebleu.sentence_ter(p, [r]).score
    for p, r in zip(preds, refs)
]
print(f"TER mean (sentence avg): {np.mean(sent_ter):.2f}")
print(f"TER std                : {np.std(sent_ter):.2f}")

TER (corpus): 42.95
TER mean (sentence avg): 42.88
TER std                : 31.42


In [21]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell M4 (fixed) — BERTScore without the bert_score package  ║
# ║  Implements the algorithm directly using transformers.        ║
# ╚══════════════════════════════════════════════════════════════╝
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
# deberta-large is slightly better correlated with human judgments than
# roberta-large for English MT evaluation, and is available on HuggingFace.
# Swap for "roberta-large" if you prefer the standard BERTScore default.
BERTSCORE_MODEL = "microsoft/deberta-large-mnli"

# Layer to extract embeddings from.
# For deberta-large (24 layers), layer 18 is optimal per the BERTScore paper.
# For roberta-large (24 layers), use layer 17.
OPTIMAL_LAYER = 18

BATCH_SIZE = 16
MAX_LENGTH = 128

print(f"Loading BERTScore backbone: {BERTSCORE_MODEL}")
bs_tokenizer = AutoTokenizer.from_pretrained(BERTSCORE_MODEL)
bs_model     = AutoModel.from_pretrained(BERTSCORE_MODEL, output_hidden_states=True)
bs_model.to(device)
bs_model.eval()
print("Model loaded.")


def get_embeddings(texts: list[str], batch_size: int = BATCH_SIZE) -> list[torch.Tensor]:
    """
    Returns a list of tensors, one per text.
    Each tensor shape: (num_tokens, hidden_size) — special tokens excluded.
    Embeddings are L2-normalised (required for cosine sim via dot product).
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        enc = bs_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(device)

        with torch.no_grad():
            out = bs_model(**enc)

        # Extract the chosen hidden layer
        hidden = out.hidden_states[OPTIMAL_LAYER]  # (B, T, H)

        for j in range(len(batch)):
            # Mask out padding and special tokens ([CLS], [SEP] / [PAD])
            mask    = enc["attention_mask"][j].bool()
            tokens  = bs_tokenizer.convert_ids_to_tokens(
                enc["input_ids"][j].tolist()
            )
            # Special token ids vary by model; use the attention mask +
            # filter out tokens that are definitively special
            special = set(bs_tokenizer.all_special_tokens)
            valid   = [
                k for k, (m, t) in enumerate(zip(mask.tolist(), tokens))
                if m and t not in special
            ]
            if not valid:
                # Fallback: use all non-padding tokens
                valid = mask.nonzero(as_tuple=True)[0].tolist()

            emb = hidden[j][valid]             # (num_valid_tokens, H)
            emb = F.normalize(emb, p=2, dim=1) # L2-normalise each token
            all_embeddings.append(emb.cpu())

    return all_embeddings


def bertscore_from_embeddings(
    pred_embs: list[torch.Tensor],
    ref_embs:  list[torch.Tensor],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Given pre-computed L2-normalised token embeddings, computes:
      Precision = mean over pred tokens of max cosine sim to any ref token
      Recall    = mean over ref  tokens of max cosine sim to any pred token
      F1        = harmonic mean of P and R
    Returns three numpy arrays of shape (N,).
    """
    P_scores, R_scores, F1_scores = [], [], []

    for pred_e, ref_e in zip(pred_embs, ref_embs):
        # Cosine similarity matrix (M_pred x M_ref)
        # Since embeddings are L2-normalised, dot product = cosine similarity
        sim = torch.mm(pred_e, ref_e.T)  # (M_pred, M_ref)

        precision = sim.max(dim=1).values.mean().item()  # best ref match per pred token
        recall    = sim.max(dim=0).values.mean().item()  # best pred match per ref token

        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = 0.0

        P_scores.append(precision)
        R_scores.append(recall)
        F1_scores.append(f1)

    return (
        np.array(P_scores),
        np.array(R_scores),
        np.array(F1_scores),
    )


def baseline_rescale(
    scores: np.ndarray,
    model_name: str,
    metric: str = "f1",
) -> np.ndarray:
    """
    Rescales BERTScore to roughly [0, 1] using the baseline from the
    BERTScore paper (computed on randomly paired sentences).
    Baselines below are for deberta-large-mnli and roberta-large on English.
    If your model isn't listed, set rescale=False.
    """
    baselines = {
        "microsoft/deberta-large-mnli": {"p": 0.7254, "r": 0.7185, "f1": 0.7219},
        "roberta-large"               : {"p": 0.8474, "r": 0.8474, "f1": 0.8474},
    }
    if model_name not in baselines:
        print(f"  No baseline for {model_name}, returning raw scores.")
        return scores
    b = baselines[model_name][metric]
    return (scores - b) / (1 - b)


# ── Run BERTScore ─────────────────────────────────────────────────────────────
print(f"\nEncoding {len(preds)} predictions...")
pred_embs = get_embeddings(preds)

print(f"Encoding {len(refs)} references...")
ref_embs  = get_embeddings(refs)

print("Computing P / R / F1...")
P_raw, R_raw, F1_raw = bertscore_from_embeddings(pred_embs, ref_embs)

# Rescale to [0, 1] then multiply by 100 for readability
P_scaled  = baseline_rescale(P_raw,  BERTSCORE_MODEL, "p")
R_scaled  = baseline_rescale(R_raw,  BERTSCORE_MODEL, "r")
F1_scaled = baseline_rescale(F1_raw, BERTSCORE_MODEL, "f1")

bs_precision = float(P_scaled.mean()  * 100)
bs_recall    = float(R_scaled.mean()  * 100)
bs_f1        = float(F1_scaled.mean() * 100)

print(f"\nBERTScore Precision (rescaled) : {bs_precision:.2f}")
print(f"BERTScore Recall    (rescaled) : {bs_recall:.2f}")
print(f"BERTScore F1        (rescaled) : {bs_f1:.2f}")

# Keep per-sentence F1 for Cell M6 bootstrap CIs
sent_bertscore_f1 = F1_scaled * 100

Loading BERTScore backbone: microsoft/deberta-large-mnli


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    revision=sha,
      ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    pr = previous_pr(api, model_id, pr_title, token=token)
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 48, in spawn_conversion
    event_id = result["event_id"]
             ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

DebertaModel LOAD REPORT from: microsoft/deberta-large-mnli
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.bias   | UNEXPECTED |  | 
config              | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 
classifier.bias     | UNEXPECTED |  | 
classifier.weight   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.

Encoding 660 predictions...
Encoding 660 references...
Computing P / R / F1...

BERTScore Precision (rescaled) : 42.19
BERTScore Recall    (rescaled) : 49.31
BERTScore F1        (rescaled) : 45.48


In [26]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell M6 — Bootstrap confidence intervals                    ║
# ║  Non-negotiable with ~132 samples. Reports 95% CI for every  ║
# ║  metric using 1000 bootstrap resamples.                      ║
# ╚══════════════════════════════════════════════════════════════╝
import sacrebleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer as rouge_lib
from jiwer import wer, cer

# We need per-sentence scores for all metrics to enable bootstrap resampling.
# Recompute them here for the full set (they may already exist from earlier cells).

rouge_scorer_obj = rouge_lib.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=False
)

def compute_per_sentence_scores(preds, refs):
    """Returns a dict of lists, one score per sample for each metric."""
    scores = {
        "bleu"     : [],
        "chrf"     : [],
        "rouge1"   : [],
        "rouge2"   : [],
        "rougeL"   : [],
        "meteor"   : [],
        "wer"      : [],
        "cer"      : [],
        "ter"      : [],
    }
    for p, r in zip(preds, refs):
        scores["bleu"].append(
            sacrebleu.sentence_bleu(p, [r]).score
        )
        scores["chrf"].append(
            sacrebleu.sentence_chrf(p, [r], word_order=2).score
        )
        rouge_out = rouge_scorer_obj.score(r, p)
        scores["rouge1"].append(rouge_out["rouge1"].fmeasure * 100)
        scores["rouge2"].append(rouge_out["rouge2"].fmeasure * 100)
        scores["rougeL"].append(rouge_out["rougeL"].fmeasure * 100)

        pt = p.split(); rt = r.split()
        scores["meteor"].append(
            meteor_score([rt], pt) * 100 if pt and rt else 0.0
        )
        scores["wer"].append(wer(r, p) * 100)
        scores["cer"].append(cer(r, p) * 100)
        scores["ter"].append(
            sacrebleu.sentence_ter(p, [r]).score
        )
    return scores

print("Computing per-sentence scores...")
sent_scores = compute_per_sentence_scores(preds, refs)

# Add BERTScore and COMET (already computed per-sentence above)
sent_scores["bertscore_f1"] = sent_bertscore_f1.tolist()
# sent_scores["comet"]        = sent_comet


def bootstrap_ci(scores, n_resamples=1000, ci=95, metric_is_corpus_bleu=False):
    """
    Bootstrap 95% CI for the mean of per-sentence scores.
    Returns (mean, lower, upper).

    Note: corpus BLEU is not simply the mean of sentence BLEUs,
    but for CI estimation this approximation is standard practice
    given the small test set size.
    """
    scores = np.array(scores)
    n      = len(scores)
    rng    = np.random.default_rng(42)

    boot_means = np.array([
        rng.choice(scores, size=n, replace=True).mean()
        for _ in range(n_resamples)
    ])

    alpha = (100 - ci) / 2
    lower = np.percentile(boot_means, alpha)
    upper = np.percentile(boot_means, 100 - alpha)
    mean  = scores.mean()
    return mean, lower, upper


print("\nBootstrap 95% CIs (1000 resamples):")
print(f"{'Metric':20s}  {'Mean':>8s}  {'95% CI':>20s}  {'±':>6s}")
print("-" * 60)

ci_results = {}
for metric, scores in sent_scores.items():
    mean, lo, hi = bootstrap_ci(scores)
    half_width   = (hi - lo) / 2
    ci_results[metric] = {
        "mean": round(mean, 2),
        "ci_lower": round(lo, 2),
        "ci_upper": round(hi, 2),
        "half_width": round(half_width, 2),
    }
    print(f"  {metric:20s}  {mean:8.2f}  [{lo:7.2f}, {hi:7.2f}]  ±{half_width:5.2f}")

Computing per-sentence scores...

Bootstrap 95% CIs (1000 resamples):
Metric                    Mean                95% CI       ±
------------------------------------------------------------
  bleu                     43.75  [  41.34,   46.25]  ± 2.46
  chrf                     65.14  [  63.32,   67.07]  ± 1.88
  rouge1                   73.76  [  72.15,   75.60]  ± 1.73
  rouge2                   55.02  [  52.67,   57.34]  ± 2.33
  rougeL                   70.70  [  68.94,   72.66]  ± 1.86
  meteor                   66.38  [  64.65,   68.25]  ± 1.80
  wer                      46.37  [  43.54,   48.95]  ± 2.70
  cer                      34.83  [  32.42,   36.96]  ± 2.27
  ter                      42.88  [  40.42,   45.22]  ± 2.40
  bertscore_f1             45.48  [  42.37,   49.02]  ± 3.33


In [29]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell M7 — Per-dialect metric table with bootstrap CIs       ║
# ╚══════════════════════════════════════════════════════════════╝

df_scores = pd.DataFrame({
    "dialect"      : dialects,
    "pred"         : preds,
    "ref"          : refs,
    **{k: v for k, v in sent_scores.items()}
})

# Metrics to include in the table
TABLE_METRICS = ["bleu", "chrf", "rouge1", "rougeL",
                 "meteor", "wer", "cer", "bertscore_f1"]

rows = []

for dialect in sorted(df_scores["dialect"].unique()):
    sub   = df_scores[df_scores["dialect"] == dialect]
    row   = {"dialect": dialect, "n": len(sub)}

    for metric in TABLE_METRICS:
        vals = sub[metric].tolist()
        if len(vals) < 5:
            # Too few samples for meaningful CI
            row[metric]         = round(np.mean(vals), 2)
            row[f"{metric}_ci"] = "n/a"
        else:
            mean, lo, hi = bootstrap_ci(vals)
            row[metric]         = round(mean, 2)
            row[f"{metric}_ci"] = f"[{lo:.1f}, {hi:.1f}]"

    rows.append(row)

# Add an "All" row for the combined set
all_row = {"dialect": "ALL", "n": len(df_scores)}
for metric in TABLE_METRICS:
    mean, lo, hi = bootstrap_ci(df_scores[metric].tolist())
    all_row[metric]         = round(mean, 2)
    all_row[f"{metric}_ci"] = f"[{lo:.1f}, {hi:.1f}]"
rows.append(all_row)

df_table = pd.DataFrame(rows)

# Print compact table (score only, no CI column)
print("\nPer-dialect metric summary (mean scores):")
display_cols = ["dialect", "n"] + TABLE_METRICS
print(df_table[display_cols].to_string(index=False))

# Print with CIs for key metrics
print("\nPer-dialect BLEU and BERTScore with 95% CIs:")
ci_cols = ["dialect", "n", "bleu", "bleu_ci", "bertscore_f1", "bertscore_f1_ci"]
print(df_table[ci_cols].to_string(index=False))

# Save full table
df_table.to_csv("/kaggle/working/metrics_per_dialect.csv", index=False)
print("\nFull table saved to: /kaggle/working/metrics_per_dialect.csv")


Per-dialect metric summary (mean scores):
   dialect   n  bleu  chrf  rouge1  rougeL  meteor   wer   cer  bertscore_f1
   Barisal 131 47.25 67.72   75.97   72.64   69.18 43.41 32.25         51.18
Chittagong 130 39.62 62.96   72.14   69.31   63.31 49.79 36.58         39.97
Mymensingh 130 48.77 70.56   78.22   75.27   70.89 39.77 29.73         57.57
  Noakhali 133 41.22 63.26   71.53   68.25   63.93 50.22 38.39         42.42
    Sylhet 136 42.02 61.39   71.11   68.18   64.73 48.48 37.01         36.67
       ALL 660 43.75 65.14   73.76   70.70   66.38 46.37 34.83         45.48

Per-dialect BLEU and BERTScore with 95% CIs:
   dialect   n  bleu      bleu_ci  bertscore_f1 bertscore_f1_ci
   Barisal 131 47.25 [41.7, 53.3]         51.18    [45.3, 57.4]
Chittagong 130 39.62 [34.6, 44.6]         39.97    [32.8, 46.7]
Mymensingh 130 48.77 [43.5, 54.5]         57.57    [51.1, 63.5]
  Noakhali 133 41.22 [35.9, 46.3]         42.42    [34.9, 49.4]
    Sylhet 136 42.02 [36.5, 47.4]         36.67    [

In [30]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell M8 — Classifier routing impact analysis                ║
# ║  Answers: how much does BLEU drop when the classifier        ║
# ║  routes a sentence to the wrong adapter?                     ║
# ╚══════════════════════════════════════════════════════════════╝

df_scores["dialect_match"] = df["dialect_match"].values  # from Cell 9

correct   = df_scores[df_scores["dialect_match"] == True]
incorrect = df_scores[df_scores["dialect_match"] == False]

print("=" * 55)
print("  ROUTING IMPACT ANALYSIS")
print("=" * 55)
print(f"  Correctly routed   : {len(correct)} samples")
print(f"  Incorrectly routed : {len(incorrect)} samples")
print()

for metric in ["bleu", "chrf", "bertscore_f1"]:
    c_mean = correct[metric].mean()
    w_mean = incorrect[metric].mean() if len(incorrect) > 0 else float("nan")
    delta  = c_mean - w_mean
    print(f"  {metric:15s}  correct={c_mean:6.2f}  "
          f"wrong={w_mean:6.2f}  Δ={delta:+.2f}")

print()

# Per-dialect misrouting breakdown
print("  Misrouting confusion (ground_truth → predicted_dialect):")
misrouted = df[df["dialect_match"] == False]
if len(misrouted) > 0:
    confusion = (
        misrouted.groupby(["ground_truth_dialect", "dialect"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    print(confusion.to_string(index=False))
else:
    print("  No misrouted samples — perfect classifier routing!")

print("=" * 55)

  ROUTING IMPACT ANALYSIS
  Correctly routed   : 637 samples
  Incorrectly routed : 23 samples

  bleu             correct= 43.84  wrong= 41.45  Δ=+2.39
  chrf             correct= 65.36  wrong= 58.96  Δ=+6.40
  bertscore_f1     correct= 45.98  wrong= 31.42  Δ=+14.56

  Misrouting confusion (ground_truth → predicted_dialect):
ground_truth_dialect         dialect  count
              Sylhet      Mymensingh      5
              Sylhet        Noakhali      5
              Sylhet Standard Bangla      4
          Chittagong        Noakhali      3
             Barisal      Mymensingh      1
          Mymensingh Standard Bangla      1
            Noakhali      Mymensingh      1
            Noakhali      Chittagong      1
          Mymensingh          Sylhet      1
              Sylhet         Barisal      1


In [31]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell M9 — Length ratio analysis                             ║
# ║  Diagnoses length_penalty issues per dialect.                ║
# ║  Ratio > 1 → model over-generates vs reference.             ║
# ║  Ratio < 1 → model under-generates (too short).             ║
# ╚══════════════════════════════════════════════════════════════╝

df_scores["pred_len"] = df_scores["pred"].str.split().str.len()
df_scores["ref_len"]  = df_scores["ref"].str.split().str.len()
df_scores["len_ratio"] = df_scores["pred_len"] / df_scores["ref_len"].clip(lower=1)

print("Length ratio analysis (pred_words / ref_words):")
print(f"  {'Ideal ratio = 1.00':}")
print()
print(f"  {'Dialect':15s}  {'Mean ratio':>12s}  {'Std':>8s}  {'Median':>8s}")
print("  " + "-" * 50)

for dialect in sorted(df_scores["dialect"].unique()):
    sub   = df_scores[df_scores["dialect"] == dialect]["len_ratio"]
    print(f"  {dialect:15s}  {sub.mean():12.3f}  {sub.std():8.3f}  {sub.median():8.3f}")

overall = df_scores["len_ratio"]
print(f"  {'ALL':15s}  {overall.mean():12.3f}  {overall.std():8.3f}  {overall.median():8.3f}")
print()

# Flag any dialect where ratio is notably off
for dialect in sorted(df_scores["dialect"].unique()):
    sub   = df_scores[df_scores["dialect"] == dialect]["len_ratio"]
    ratio = sub.mean()
    if ratio > 1.15:
        print(f"  ⚠️  {dialect}: over-generating "
              f"(ratio={ratio:.2f}). Consider increasing length_penalty.")
    elif ratio < 0.85:
        print(f"  ⚠️  {dialect}: under-generating "
              f"(ratio={ratio:.2f}). Consider decreasing length_penalty.")
    else:
        print(f"  ✅  {dialect}: length ratio OK ({ratio:.2f})")

Length ratio analysis (pred_words / ref_words):
  Ideal ratio = 1.00

  Dialect            Mean ratio       Std    Median
  --------------------------------------------------
  Barisal                 1.017     0.187     1.000
  Chittagong              1.024     0.182     1.000
  Mymensingh              1.025     0.170     1.000
  Noakhali                1.043     0.198     1.000
  Sylhet                  1.032     0.193     1.000
  ALL                     1.028     0.186     1.000

  ✅  Barisal: length ratio OK (1.02)
  ✅  Chittagong: length ratio OK (1.02)
  ✅  Mymensingh: length ratio OK (1.02)
  ✅  Noakhali: length ratio OK (1.04)
  ✅  Sylhet: length ratio OK (1.03)


In [32]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell M10 — Final consolidated report                        ║
# ╚══════════════════════════════════════════════════════════════╝
import json
from pathlib import Path

report = {
    "test_set_size" : len(df),
    "corpus_metrics": {
        "sacrebleu" : round(sacrebleu.corpus_bleu(preds, [refs]).score, 4),
        "chrfpp"    : round(sacrebleu.corpus_chrf(preds, [refs], word_order=2).score, 4),
        "ter"       : round(ter_score, 4),
        "bertscore_f1" : round(bs_f1, 4),
    },
    "bootstrap_95ci": ci_results,
    "per_dialect"   : df_table.set_index("dialect").to_dict(orient="index"),
    "routing": {
        "correctly_routed"   : int(len(correct)),
        "incorrectly_routed" : int(len(incorrect)),
        "routing_accuracy"   : round(len(correct) / len(df), 4),
    },
}

report_path = Path("/kaggle/working/full_eval_report.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2, default=str)

print("=" * 60)
print("  FULL EVALUATION REPORT")
print("=" * 60)
print(f"\n  Test samples       : {report['test_set_size']}")
print(f"\n  Corpus-level metrics:")
for k, v in report["corpus_metrics"].items():
    ci  = ci_results.get(k.replace("sacrebleu", "bleu"), {})
    hw  = ci.get("half_width", "—")
    print(f"    {k:20s}: {v:7.2f}  (±{hw})")
print(f"\n  Classifier routing accuracy: "
      f"{report['routing']['routing_accuracy']:.2%}")
print(f"\nFull report saved to: {report_path}")

  FULL EVALUATION REPORT

  Test samples       : 660

  Corpus-level metrics:
    sacrebleu           :   43.37  (±2.46)
    chrfpp              :   64.81  (±—)
    ter                 :   42.95  (±2.4)
    bertscore_f1        :   45.48  (±3.33)

  Classifier routing accuracy: 96.52%

Full report saved to: /kaggle/working/full_eval_report.json
